## Методические указания по выполнению практикума №3

[МУ блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

**Тема: Исследование и имплементация SOTA-моделей на примере семейства ViT в экосистеме HuggingFace; комбинирование моделей в комплексные решения**

**Тема РПД:** Л13/П3. **Индикатор:** DL-3.2, уровень П.

**Ноутбук:** `#6 P3_ViT_HuggingFace.ipynb`. Практикум опирается на навыки дообучения, полученные в ЛР1–ЛР2, но переносит их в другую инженерную среду: вместо ручного цикла обучения на `torchvision` используется экосистема HuggingFace.

**Цель работы:** освоить рабочий цикл применения SOTA-моделей семейства ViT средствами экосистемы HuggingFace (Hub, `pipeline`, `AutoImageProcessor`, `AutoModelForImageClassification`, `Trainer`) и собрать из нескольких моделей составной пайплайн, решающий задачу, которую одна модель не решает.

**Задачи:**

- Разобрать устройство ViT и следствия, важные для практики: фиксированное разрешение входа, привязка препроцессинга к весам, роль CLS-токена.
- Получить baseline готовым `pipeline` на предобученных весах ImageNet и зафиксировать границу его применимости.
- Воспроизвести тот же инференс вручную через `AutoImageProcessor` и `AutoModelForImageClassification`, сравнив время работы.
- Дообучить ViT под целевые классы через `Trainer`, выбрав checkpoint по validation.
- Спроектировать и реализовать составной пайплайн: маршрутизация + ансамбль экспертов + правило отказа.
- Провести сопоставимое сравнение всех конфигураций по качеству, времени и числу обучаемых параметров и сделать ограниченный вывод.

**Что этот практикум не повторяет.** Исследование «linear probe против fine-tuning» и сравнение индуктивных смещений ViT и ConvNeXt выполняется в ЛР блока 4. Здесь предмет изучения — инструментальная среда и композиция моделей, а не выбор стратегии переноса обучения.

### 1. Теоретическая часть

#### 1.1 Vision Transformer: устройство и практические следствия

ViT ([Dosovitskiy et al., 2020](https://arxiv.org/abs/2010.11929)) отказывается от свёрточного индуктивного смещения. Изображение $x \in \mathbb{R}^{H \times W \times C}$ разрезается на неперекрывающиеся патчи размера $P \times P$, число патчей

$$N = \frac{H W}{P^{2}}.$$

Каждый патч разворачивается в вектор и линейно проецируется в пространство размерности $D$ (patch embedding). К последовательности слева добавляется обучаемый служебный токен `[CLS]`, и ко всем позициям прибавляются позиционные эмбеддинги $E_{pos}$:

$$z_0 = [\, x_{\text{cls}};\; x^1_p E;\; x^2_p E;\; \dots;\; x^N_p E \,] + E_{pos}.$$

Далее последовательность обрабатывается блоками трансформера, ядро которых — многоголовое самовнимание:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V .$$

Классификация выполняется головой поверх выходного состояния `[CLS]`.

Три следствия, которые нужно держать в голове при работе с готовыми весами:

1. **Разрешение входа фиксировано весами.** Число позиционных эмбеддингов равно $N + 1$. Смена разрешения требует интерполяции позиционных эмбеддингов, иначе веса неприменимы. Имя чекпоинта обычно кодирует и патч, и разрешение: `vit-base-patch16-224`.
2. **Препроцессинг — часть модели.** Размер, способ ресайза, `image_mean` и `image_std` заданы при обучении чекпоинта. Несовпадение нормализации даёт «необъяснимо низкое» качество — типичная ошибка, отдельно отмеченная в рубрике оценивания.
3. **ViT слабее свёрточных сетей при малом объёме данных.** Отсутствие локального смещения компенсируется предобучением; поэтому в прикладной задаче почти всегда используется предобученный чекпоинт, а не обучение с нуля.

Семейство: DeiT (дистилляционный токен и режим обучения без огромного датасета), Swin (окна и иерархия), MobileViT (гибрид для мобильных устройств), ConvNeXt (свёрточный ответ на трансформеры).

#### 1.2 Экосистема HuggingFace

Экосистема состоит из нескольких слоёв, и выбор слоя — инженерное решение.

| Слой | API | Когда применять | Чем ограничен |
|---|---|---|---|
| Готовый пайплайн | `pipeline("image-classification", model=...)` | быстрая проверка гипотезы, демонстрация | скрывает препроцессинг и батчинг, метки — исходные метки чекпоинта |
| Явная пара процессор + модель | `AutoImageProcessor`, `AutoModelForImageClassification` | инференс в проде, батчевая обработка, доступ к логитам | требует ручного управления устройством и батчами |
| Обучение | `TrainingArguments`, `Trainer` | дообучение, оценка, checkpointing | свои соглашения о формате датасета |

Ключевые соглашения, о которые чаще всего спотыкаются:

- модель ожидает словарь с ключом `pixel_values`, целевые метки передаются в ключе `labels`;
- при смене числа классов голова не совпадает с весами чекпоинта, поэтому требуется `ignore_mismatched_sizes=True` и явные словари `id2label` / `label2id` — иначе результат инференса нечитаем;
- `Trainer` выбирает лучший checkpoint по метрике на `eval_dataset`; тестовый набор в `Trainer` не передаётся.

#### 1.3 Комбинирование моделей в комплексное решение

Одна модель редко закрывает прикладную задачу целиком. Базовые схемы композиции:

- **Каскад (маршрутизация).** Первая модель отвечает на грубый вопрос и направляет объект нужному «эксперту». Экономит вычисления и позволяет обучать экспертов на своих подзадачах.
- **Ансамбль.** Несколько моделей решают одну задачу, их выходы агрегируются. Для вероятностей — взвешенное усреднение

$$p(c \mid x) = \sum_{i=1}^{M} w_i \, p_i(c \mid x), \qquad \sum_i w_i = 1, \; w_i \ge 0 .$$

  Усреднять следует вероятности после `softmax`, а не логиты: логиты разных моделей имеют разный масштаб.
- **Правило отказа (reject option).** Если уверенность ниже порога $\tau$, система не выдаёт ответ, а передаёт объект дальше (другой модели или человеку):

$$\hat{y}(x) = \begin{cases} \arg\max_c p(c \mid x), & \max_c p(c \mid x) \ge \tau, \\ \varnothing, & \text{иначе.} \end{cases}$$

  Порог $\tau$ подбирается **по validation** и превращает одну точку качества в кривую «покрытие — точность» (accuracy при доле обработанных объектов).

Стоимость композиции не бесплатна: время инференса складывается, а прирост качества обычно сублинеен. Поэтому составной пайплайн оценивается по двум осям сразу — качество и время.

#### 1.4 Методика сопоставимого сравнения

Сравнение конфигураций корректно, только если совпадают: разбиение данных, разрешение и препроцессинг (с точностью до того, что предписано весами), набор аугментаций, число эпох и критерий выбора checkpoint, аппаратное окружение при замере времени. Меняется ровно один фактор за серию. Выбор любых порогов и гиперпараметров — по validation; test используется один раз после фиксации решения.

### 2. Практическая часть

#### 2.1 Подготовка окружения

Работа рассчитана на один бесплатный GPU (Google Colab / Kaggle). Установите зависимости с закреплёнными мажорными версиями — незакреплённые версии `transformers` регулярно меняют имена аргументов `TrainingArguments`, и ноутбук перестаёт воспроизводиться.

In [ ]:
# Версии закреплены по мажорной компоненте: незакреплённая установка ломает воспроизводимость.
%pip install -q "torch>=2.2,<3.0" "torchvision>=0.17,<1.0" "transformers>=4.44,<5.0" "accelerate>=0.33,<2.0" "timm>=1.0,<2.0" "scikit-learn>=1.4,<2.0" "pandas>=2.0,<3.0" "matplotlib>=3.8,<4.0"

In [ ]:
import json
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

import torchvision
import transformers
from torchvision.datasets import OxfordIIITPet
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
    pipeline,
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

SEED = 42
DATA_ROOT = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", DEVICE)
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

Служебные функции ниже даны готовыми: фиксация seed, журнал экспериментов и сводная таблица. Журнал пишется в `outputs/runs.jsonl` — без него после нескольких запусков невозможно восстановить, при каких условиях получен результат.

In [ ]:
def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


RUNS_PATH = OUTPUT_DIR / "runs.jsonl"
RUNS: list[dict] = []


def log_run(name: str, **fields) -> dict:
    """Записать эксперимент в журнал.

    Обязательные поля фиксируются автоматически, остальные передаются вызовом.
    """
    record = {"name": name, "seed": SEED, "device": DEVICE.type, **fields}
    RUNS.append(record)
    with RUNS_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record


def results_table(columns: list[str] | None = None) -> pd.DataFrame:
    df = pd.DataFrame(RUNS)
    if columns:
        columns = [c for c in columns if c in df.columns]
        df = df[columns]
    return df


def evaluate_predictions(y_true, y_pred) -> dict:
    """accuracy и macro F1 — единые метрики для всех конфигураций работы."""
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


set_global_seed()

#### 2.2 Данные

Используется Oxford-IIIT Pet: 37 пород кошек и собак, около 7400 изображений. Набор удобен тем, что классы визуально близки — готовый ImageNet-классификатор на нём заметно ошибается, и разница между baseline и дообученной моделью хорошо видна.

**Параметр `N_SUBSET`** задаёт размер обучающей подвыборки. Уменьшение выборки нужно по трём причинам: (1) практикум рассчитан на 2 академических часа и один бесплатный GPU; (2) при отладке пайплайна полный прогон обучения бессмысленно дорог; (3) фиксированный небольшой размер выборки делает сравнение конфигураций сопоставимым по бюджету данных. Итоговые значения метрик при этом ниже, чем на полном наборе, — это ограничение обязательно указывается в выводе.

Разбиение фиксируется один раз функцией со стратификацией и seed. Повторное случайное разбиение между сериями недопустимо: оно разрушает сопоставимость.

In [ ]:
N_SUBSET = 1480      # изображений в train (примерно 40 на класс)
N_VAL = 740          # изображений в validation
N_TEST = 740         # изображений в test
IMAGE_SIZE = 224
BATCH_SIZE = 32

trainval_ds = OxfordIIITPet(root=DATA_ROOT, split="trainval",
                            target_types="category", download=True)
test_ds = OxfordIIITPet(root=DATA_ROOT, split="test",
                        target_types="category", download=True)

CLASS_NAMES = list(trainval_ds.classes)
NUM_CLASSES = len(CLASS_NAMES)
ID2LABEL = {i: name for i, name in enumerate(CLASS_NAMES)}
LABEL2ID = {name: i for i, name in ID2LABEL.items()}

print("классов:", NUM_CLASSES, "| trainval:", len(trainval_ds), "| test:", len(test_ds))
print("первые классы:", CLASS_NAMES[:5])


def dataset_labels(dataset) -> np.ndarray:
    """Метки датасета без чтения изображений (иначе стратификация слишком дорога)."""
    for attr in ("_labels", "targets", "labels"):
        values = getattr(dataset, attr, None)
        if values is not None:
            return np.asarray(values)
    return np.asarray([dataset[i][1] for i in range(len(dataset))])


def stratified_indices(labels: np.ndarray, n_total: int, seed: int = SEED,
                       exclude: set[int] | None = None) -> list[int]:
    """Отобрать n_total индексов, сохраняя пропорции классов."""
    rng = np.random.default_rng(seed)
    exclude = exclude or set()
    per_class = max(1, n_total // len(np.unique(labels)))
    chosen: list[int] = []
    for cls in np.unique(labels):
        pool = [i for i in np.flatnonzero(labels == cls) if i not in exclude]
        rng.shuffle(pool)
        chosen.extend(pool[:per_class])
    rng.shuffle(chosen)
    return [int(i) for i in chosen[:n_total]]

In [ ]:
trainval_labels = dataset_labels(trainval_ds)
test_labels = dataset_labels(test_ds)

train_idx = stratified_indices(trainval_labels, N_SUBSET, seed=SEED)
val_idx = stratified_indices(trainval_labels, N_VAL, seed=SEED + 1, exclude=set(train_idx))
test_idx = stratified_indices(test_labels, N_TEST, seed=SEED + 2)

assert not (set(train_idx) & set(val_idx)), "train и validation пересекаются"

train_base = Subset(trainval_ds, train_idx)
val_base = Subset(trainval_ds, val_idx)
test_base = Subset(test_ds, test_idx)

SPLIT_SIZES = {"train": len(train_base), "val": len(val_base), "test": len(test_base)}
print(SPLIT_SIZES)

# TODO (задание 1.1): проверьте сбалансированность полученных подвыборок.
# Постройте распределение классов в train/val/test и убедитесь, что ни один класс
# не потерян. Если классы представлены крайне неравномерно, macro F1 и accuracy
# разойдутся — отметьте это в отчёте.

#### 2.3 Baseline: готовый пайплайн HuggingFace

Первый шаг любой работы — минимально разумная конфигурация, относительно которой измеряются изменения. Здесь это готовый `pipeline` с чекпоинтом `google/vit-base-patch16-224`, обученным на ImageNet-1k.

Обратите внимание: этот чекпоинт предсказывает 1000 классов ImageNet, а не 37 пород целевой задачи. Часть пород присутствует в ImageNet под своими именами, часть — нет. Baseline нужен именно для того, чтобы измерить эту границу применимости, а не чтобы получить высокое качество.

In [ ]:
BASELINE_CHECKPOINT = "google/vit-base-patch16-224"

clf_pipeline = pipeline(
    task="image-classification",
    model=BASELINE_CHECKPOINT,
    device=0 if DEVICE.type == "cuda" else -1,
)

sample_image, sample_label = trainval_ds[train_idx[0]]
print("истинный класс:", CLASS_NAMES[sample_label])
for prediction in clf_pipeline(sample_image.convert("RGB"), top_k=5):
    print(f"  {prediction['label']:<40} {prediction['score']:.3f}")

**Задание 2.** Оцените baseline количественно на всём тестовом сплите, а не на нескольких «показательных» картинках.

Прямое сравнение меток ImageNet с именами пород невозможно, поэтому измеряется ослабленный критерий: **попала ли верная порода в top-5 предсказаний по совпадению названия** (нормализованному: нижний регистр, `_` заменяется пробелом). Такой критерий заведомо оптимистичен для baseline — это допустимо, поскольку baseline должен быть сильным, а не удобным.

In [ ]:
def normalize_label(text: str) -> str:
    return text.lower().replace("_", " ").replace("-", " ").strip()


def baseline_top5_match(dataset, indices, hf_pipeline, batch_size: int = 32) -> dict:
    """Доля объектов, для которых имя истинного класса встретилось в top-5 меток.

    Контракт:
        dataset, indices -- исходный датасет и индексы оцениваемого сплита;
        hf_pipeline      -- готовый transformers.pipeline для image-classification;
        возвращает dict с ключами 'match_rate' и 'inference_time_s'.

    Реализуйте батчевую обработку: pipeline принимает список изображений
    и аргумент batch_size. Замерьте время инференса time.perf_counter().
    """
    raise NotImplementedError


# TODO (задание 2.1): реализуйте baseline_top5_match и вычислите метрику на test.
# TODO (задание 2.2): запишите результат в журнал через log_run, указав checkpoint,
#                     размер сплита и время инференса.
# TODO (задание 2.3): выпишите 3-5 пород, которые baseline не распознаёт в принципе,
#                     и объясните почему (загляните в id2label чекпоинта).

#### 2.4 Явная пара «процессор — модель»

`pipeline` удобен, но скрывает препроцессинг и не даёт доступа к логитам, которые понадобятся для ансамбля. Ниже тот же инференс собирается вручную.

`AutoImageProcessor` восстанавливает препроцессинг ровно в том виде, в каком он применялся при обучении чекпоинта: посмотрите поля `size`, `image_mean`, `image_std`. Именно поэтому нельзя подставлять «свою привычную» нормализацию ImageNet, не сверившись с процессором.

In [ ]:
processor = AutoImageProcessor.from_pretrained(BASELINE_CHECKPOINT)
print("size:", processor.size)
print("mean:", processor.image_mean, "| std:", processor.image_std)


class HFImageDataset(torch.utils.data.Dataset):
    """Обёртка torchvision-датасета в формат, ожидаемый моделями transformers.

    Возвращает словарь {'pixel_values': FloatTensor[C,H,W], 'labels': int}.
    Аугментации (PIL -> PIL) применяются ДО процессора, чтобы не нарушить
    предписанную чекпоинтом нормализацию.
    """

    def __init__(self, base, image_processor, augment=None):
        self.base = base
        self.processor = image_processor
        self.augment = augment

    def __len__(self) -> int:
        return len(self.base)

    def __getitem__(self, index: int) -> dict:
        image, label = self.base[index]
        image = image.convert("RGB")
        if self.augment is not None:
            image = self.augment(image)
        encoded = self.processor(images=image, return_tensors="pt")
        return {"pixel_values": encoded["pixel_values"][0], "labels": int(label)}


def collate_fn(batch: list[dict]) -> dict:
    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "labels": torch.tensor([item["labels"] for item in batch], dtype=torch.long),
    }


@torch.no_grad()
def predict_probabilities(model, dataset, batch_size: int = BATCH_SIZE):
    """Вернуть (вероятности [N, C], метки [N], время инференса в секундах)."""
    model.eval().to(DEVICE)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    probabilities, targets = [], []
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    for batch in loader:
        logits = model(pixel_values=batch["pixel_values"].to(DEVICE)).logits
        probabilities.append(torch.softmax(logits, dim=-1).cpu().numpy())
        targets.append(batch["labels"].numpy())
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    return np.concatenate(probabilities), np.concatenate(targets), elapsed

#### 2.5 Дообучение через `Trainer`

Голова чекпоинта рассчитана на 1000 классов ImageNet, целевая задача — 37 классов. При загрузке модели нужно передать новое число классов, словари меток и разрешить несовпадение размеров головы. Без `ignore_mismatched_sizes=True` загрузка завершится ошибкой; без `id2label` предсказания останутся безымянными.

Для дообучения берётся чекпоинт, предобученный на ImageNet-21k (`google/vit-base-patch16-224-in21k`): у него голова не «зашита» под 1000 классов ImageNet-1k, и перенос на новый набор классов чище.

Имя аргумента стратегии валидации в `TrainingArguments` менялось между версиями (`evaluation_strategy` → `eval_strategy`). Ячейка ниже определяет актуальное имя по сигнатуре, чтобы ноутбук воспроизводился на разных сборках.

In [ ]:
import inspect

FINETUNE_CHECKPOINT = "google/vit-base-patch16-224-in21k"
finetune_processor = AutoImageProcessor.from_pretrained(FINETUNE_CHECKPOINT)

train_hf = HFImageDataset(train_base, finetune_processor)
val_hf = HFImageDataset(val_base, finetune_processor)
test_hf = HFImageDataset(test_base, finetune_processor)


def build_classifier(checkpoint: str):
    """Загрузить AutoModelForImageClassification под NUM_CLASSES классов."""
    return AutoModelForImageClassification.from_pretrained(
        checkpoint,
        num_labels=NUM_CLASSES,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )


def compute_metrics(eval_prediction) -> dict:
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return evaluate_predictions(labels, predictions)


_ta_signature = inspect.signature(TrainingArguments.__init__).parameters
EVAL_STRATEGY_KEY = "eval_strategy" if "eval_strategy" in _ta_signature else "evaluation_strategy"
print("аргумент стратегии валидации:", EVAL_STRATEGY_KEY)

In [ ]:
# TODO (задание 3.1): соберите TrainingArguments для дообучения.
# Обязательно задайте:
#   output_dir, num_train_epochs (2-3 при данном бюджете),
#   per_device_train_batch_size / per_device_eval_batch_size,
#   learning_rate (для дообучения ViT типичен порядок 1e-5 ... 5e-5),
#   **{EVAL_STRATEGY_KEY: "epoch"}, save_strategy="epoch",
#   load_best_model_at_end=True, metric_for_best_model="accuracy",
#   seed=SEED, report_to="none", fp16=(DEVICE.type == "cuda").
# Обоснуйте выбор learning rate: почему при дообучении он на порядок ниже,
# чем при обучении головы с нуля?

training_args = None  # TODO

# TODO (задание 3.2): создайте Trainer(model=..., args=..., train_dataset=...,
# eval_dataset=..., data_collator=collate_fn, compute_metrics=compute_metrics)
# и запустите обучение. Зафиксируйте время обучения и число обучаемых параметров.
# ВНИМАНИЕ: eval_dataset -- это validation, а не test. Test в Trainer не передаётся.

set_global_seed()
model_finetuned = build_classifier(FINETUNE_CHECKPOINT)
print("обучаемых параметров:", count_trainable_parameters(model_finetuned))

trainer = None  # TODO

In [ ]:
# TODO (задание 3.3): оцените дообученную модель на test ОДИН раз,
# после того как обучение завершено и checkpoint выбран по validation.
# Используйте predict_probabilities и evaluate_predictions.
# Результат и время инференса запишите через log_run с полями:
#   checkpoint, strategy="finetune_trainer", epochs, learning_rate,
#   trainable_params, train_time_s, inference_time_s, accuracy, macro_f1.

# TODO (задание 3.4): постройте кривые обучения по trainer.state.log_history
# (train loss и eval accuracy по эпохам) и прокомментируйте наличие или
# отсутствие признаков переобучения.

#### 2.6 Составной пайплайн

Дообученная модель работает в предположении, что на входе — кошка или собака. В реальном потоке данных это предположение нарушается. Соберите пайплайн из трёх элементов:

1. **Маршрутизатор** — baseline-классификатор ImageNet. По его top-k меткам определяется, относится ли изображение к домашним животным вообще. Если нет, объект отклоняется до вызова дорогих экспертов.
2. **Ансамбль экспертов** — дообученная ViT-модель и вторая модель того же семейства (например, `facebook/deit-small-patch16-224`), дообученная по тому же протоколу. Вероятности усредняются с весами.
3. **Правило отказа** — порог $\tau$ по максимальной вероятности ансамбля; порог подбирается по validation.

Оценивать составной пайплайн одной accuracy некорректно: при $\tau > 0$ часть объектов не обрабатывается. Считайте пару величин — **покрытие** (доля объектов, по которым выдан ответ) и **accuracy на покрытых объектах**.

In [ ]:
@dataclass
class CompositeConfig:
    router_top_k: int = 5
    reject_threshold: float = 0.0     # tau; 0.0 -- отказ отключён
    expert_weights: tuple = (0.5, 0.5)


class CompositePipeline:
    """Составной пайплайн: маршрутизация -> ансамбль экспертов -> правило отказа.

    Контракт:
        __init__(router, experts, config)
            router  -- transformers.pipeline или callable, дающий грубую метку домена;
            experts -- список моделей AutoModelForImageClassification, обученных
                       на одном и том же наборе классов и одном split;
            config  -- CompositeConfig.

        predict(dataset) -> dict с ключами:
            'predictions'   -- np.ndarray[int], -1 для отклонённых объектов;
            'probabilities' -- np.ndarray[N, NUM_CLASSES] усреднённых вероятностей;
            'coverage'      -- доля объектов с predictions != -1;
            'time_s'        -- полное время работы пайплайна.

    Требования к реализации:
        * усредняются вероятности после softmax, а не логиты;
        * веса экспертов нормируются: sum(w_i) == 1;
        * порог reject_threshold применяется к max вероятности ансамбля;
        * объект, отклонённый маршрутизатором, не подаётся экспертам.
    """

    def __init__(self, router, experts, config: CompositeConfig):
        self.router = router
        self.experts = experts
        self.config = config

    def route(self, dataset) -> np.ndarray:
        """Вернуть булеву маску: True -- объект передаётся экспертам."""
        raise NotImplementedError

    def predict(self, dataset) -> dict:
        raise NotImplementedError


# TODO (задание 4.1): реализуйте route и predict.
# TODO (задание 4.2): подберите reject_threshold ПО VALIDATION.
#   Постройте зависимость accuracy от coverage для tau из np.linspace(0.0, 0.95, 20)
#   и выберите точку, обоснованную требованиями задачи (например, accuracy >= 0.9
#   при максимально возможном покрытии). Подбор по test запрещён.

In [ ]:
# TODO (задание 4.3): дообучите второго эксперта по ТОМУ ЖЕ протоколу,
# что и первого: тот же split, то же число эпох, тот же learning rate, тот же seed.
# Отличается только checkpoint (например, "facebook/deit-small-patch16-224").
# Любое другое отличие делает вклад ансамбля неинтерпретируемым.

SECOND_EXPERT_CHECKPOINT = "facebook/deit-small-patch16-224"

# TODO (задание 4.4): оцените на test три конфигурации и запишите их в журнал:
#   а) один эксперт (дообученный ViT);
#   б) ансамбль двух экспертов без отказа (tau = 0);
#   в) полный составной пайплайн с маршрутизацией и выбранным tau.
# Для каждой зафиксируйте accuracy, macro_f1, coverage, inference_time_s.

# TODO (задание 4.5): ответьте в отчёте, окупается ли ансамбль:
# сопоставьте прирост accuracy с ростом времени инференса.

#### 2.7 Сопоставимое сравнение

Все конфигурации оценены на одном и том же тестовом сплите, с препроцессингом, предписанным соответствующим чекпоинтом, при одном seed и на одном устройстве. Сведите результаты в одну таблицу — в отчёт входит именно она, а не отдельные числа из ячеек выше.

Проверьте перед сборкой таблицы: если какая-то строка получена на другом split, при другом числе эпох или на другом устройстве, её нельзя ставить рядом с остальными без явной оговорки.

In [ ]:
SUMMARY_COLUMNS = [
    "name", "checkpoint", "strategy", "epochs", "learning_rate",
    "trainable_params", "train_time_s", "inference_time_s",
    "coverage", "accuracy", "macro_f1",
]

summary = results_table(SUMMARY_COLUMNS)
summary

# TODO (задание 5.1): дополните таблицу столбцом «время на изображение, мс»
# (inference_time_s / N_TEST * 1000).
# TODO (задание 5.2): постройте диаграмму рассеяния «accuracy vs время на изображение»
# с подписями конфигураций. Отметьте точки, лежащие на Парето-фронте.
# TODO (задание 5.3): постройте зависимость accuracy от coverage для составного
# пайплайна (по validation) и отметьте выбранный порог tau.

#### 2.8 Анализ ошибок

Средние метрики скрывают структуру ошибок. На Oxford-IIIT Pet ошибки почти всегда сосредоточены в группах близких пород, а не размазаны равномерно.

In [ ]:
def show_errors(dataset_base, probabilities: np.ndarray, targets: np.ndarray,
                class_names: list, top_n: int = 8) -> None:
    """Показать объекты с наибольшей уверенностью в неверном классе."""
    predictions = probabilities.argmax(axis=1)
    wrong = np.flatnonzero(predictions != targets)
    if wrong.size == 0:
        print("ошибок нет")
        return
    order = wrong[np.argsort(-probabilities[wrong, predictions[wrong]])][:top_n]
    columns = min(4, len(order))
    rows = int(np.ceil(len(order) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows))
    for axis, index in zip(np.atleast_1d(axes).ravel(), order):
        image, _ = dataset_base[int(index)]
        axis.imshow(image.convert("RGB"))
        axis.set_title(
            f"истина: {class_names[targets[index]]}\n"
            f"прогноз: {class_names[predictions[index]]} "
            f"({probabilities[index, predictions[index]]:.2f})",
            fontsize=9,
        )
        axis.axis("off")
    plt.tight_layout()
    plt.show()


# TODO (задание 6.1): постройте confusion matrix дообученной модели на test
# и выделите 3-5 пар классов с наибольшей взаимной путаницей.
# TODO (задание 6.2): вызовите show_errors и объясните по конкретным изображениям,
# что именно вводит модель в заблуждение (ракурс, окрас, размер объекта в кадре,
# наличие нескольких животных).
# TODO (задание 6.3): проверьте, исправляет ли ансамбль хотя бы часть этих ошибок
# или воспроизводит их. Уверенные согласованные ошибки экспертов -- признак того,
# что модели делят общее смещение, и ансамбль здесь не помогает.

### Отчёт

Отчётная часть оформляется в этом же ноутбуке и включает:

1. **Условия эксперимента:** окружение (GPU, версии `torch`/`transformers`), seed, значения `N_SUBSET`, `N_VAL`, `N_TEST`, разрешение входа, число эпох, learning rate.
2. **Сводную таблицу** всех конфигураций: baseline `pipeline`, дообученная модель, второй эксперт, ансамбль, полный составной пайплайн — с accuracy, macro F1, coverage, числом обучаемых параметров, временем обучения и временем инференса на изображение.
3. **Не менее трёх визуализаций:** кривые обучения, «accuracy vs время», зависимость accuracy от coverage.
4. **Схему составного пайплайна** (текстом или блок-схемой) с явным указанием, где принимается решение о маршрутизации и где — об отказе, и как выбран порог.
5. **Анализ ошибок:** confusion matrix, наиболее путаемые пары классов, разбор конкретных изображений.
6. **Выводы, отделённые от наблюдений.** Наблюдение — измеренный факт («ансамбль дал +1.8 п.п. accuracy при росте времени инференса в 1.9 раза»). Вывод — утверждение в пределах проверенных условий, с указанием ограничений: подвыборка `N_SUBSET`, малое число эпох, один seed, одно устройство.

Работа не засчитывается, если: порог отказа или гиперпараметры подбирались по test; конфигурации сравнивались на разных split или при разном числе эпох; приведён только лучший результат без журнала запусков.

### Контрольные вопросы

1. Как связаны размер патча, разрешение входа и число позиционных эмбеддингов в ViT? Что произойдёт при подаче изображения другого разрешения без интерполяции позиционных эмбеддингов?
2. Зачем нужен `[CLS]`-токен и чем классификация по нему отличается от усреднения выходов по всем патчам?
3. Что именно восстанавливает `AutoImageProcessor` из чекпоинта и почему нельзя использовать «стандартную» нормализацию ImageNet, не сверившись с ним?
4. Для чего нужны `ignore_mismatched_sizes=True` и словари `id2label`/`label2id` при загрузке модели под новое число классов?
5. Чем `pipeline` отличается от явной пары «процессор + модель» и в каких ситуациях `pipeline` использовать нельзя?
6. По какому набору `Trainer` выбирает лучший checkpoint и почему туда нельзя передать test?
7. Почему при усреднении в ансамбле складывают вероятности, а не логиты?
8. Как правило отказа превращает одно значение accuracy в кривую «покрытие — точность» и как выбирается порог $\tau$?
9. В каком случае ансамбль двух моделей не даёт прироста качества? Как это увидеть по анализу ошибок?
10. Какие ограничения на выводы накладывает обучение на подвыборке `N_SUBSET` при одном seed?